In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

CSV_FILE = "rahul_transactions.csv"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# ---- tiny helpers used by the whole report (formatting only) ----
def inr(x):
    # Indian digit grouping: 1234567 -> 12,34,567
    sign = "-" if x < 0 else ""
    s = str(int(round(abs(x))))
    if len(s) > 3:
        head, tail = s[:-3], s[-3:]
        parts = []
        while len(head) > 2:
            parts.insert(0, head[-2:]); head = head[:-2]
        if head: parts.insert(0, head)
        s = ",".join(parts) + "," + tail
    return f"{sign}Rs. {s}"

def bar(value, max_value, width=30, ch="#"):
    n = 0 if max_value == 0 else int(round(width * value / max_value))
    return ch * max(n, 0)

def divider(ch="=", n=64):
    return ch * n

In [5]:
import pandas as pd
import numpy as np

raw = pd.read_csv("DADS MP2 Dataset.csv", dtype=str)
print("Raw shape:", raw.shape)

# Treat empty strings as missing
raw["Mode"] = raw["Mode"].replace("", np.nan)

# Identify ISO-format dates (YYYY-MM-DD)
is_iso = (
    raw["Date"].str.len().eq(10)
    & raw["Date"].str[4].eq("-")
)

# Naive date parsing
naive = pd.to_datetime(
    raw["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

# Parse ISO dates separately
iso_part = pd.to_datetime(
    raw["Date"].where(is_iso),
    format="%Y-%m-%d",
    errors="coerce"
)

# Parse remaining dates
rest_part = pd.to_datetime(
    raw["Date"].where(~is_iso),
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Combine parsed dates
raw["date"] = iso_part.fillna(rest_part)

print(
    f"Naive dayfirst=True parse: latest date = "
    f"{naive.max().date() if naive.notna().any() else 'No valid dates'}"
)

print(
    f"Rows the naive parse got wrong: "
    f"{int((naive != raw['date']).sum())}"
)

print(
    f"Correct parse range: "
    f"{raw['date'].min()} -> {raw['date'].max()}"
)

Raw shape: (1328, 8)
Naive dayfirst=True parse: latest date = 2024-06-30
Rows the naive parse got wrong: 0
Correct parse range: 2024-01-01 00:00:00 -> 2024-06-30 00:00:00


In [6]:
raw["amount"] = pd.to_numeric(
    raw["Amount"].str.replace("₹", "", regex=False)
                 .str.replace("Rs.", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.strip(),
    errors="coerce")

# ---- TYPE: DR/Debit -> debit ; CR/Credit -> credit ----
type_map = {"dr": "debit", "debit": "debit", "cr": "credit", "credit": "credit"}
raw["type"] = raw["Type"].str.strip().str.lower().map(type_map)

# ---- Time-derived columns ----
raw["hour"] = raw["Time"].str[:2].astype(int)
raw["month"] = raw["date"].dt.month
raw["day_of_week"] = raw["date"].dt.day_name()

n_before = len(raw)
df = raw.drop_duplicates(subset=["Date", "Time", "Description", "Type", "Amount", "Balance", "Mode", "Ref"]).copy()
df = df[df["date"].notna() & df["amount"].notna() & df["type"].notna()].reset_index(drop=True)

print(f"Parsed {len(df)} transactions across {df['month'].nunique()} months. "
      f"Dropped {n_before - len(df)} duplicates. "
      f"{int(raw['amount'].isna().sum())} unparseable amounts, {int(raw['date'].isna().sum())} unparseable dates.")
print()
print(df[["date", "amount", "type", "hour", "month"]].dtypes)

Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 0 unparseable dates.

date      datetime64[ns]
amount           float64
type              object
hour               int64
month              int32
dtype: object


In [7]:
print("Distinct raw descriptions:", df["Description"].nunique())
print(sorted(df["Description"].unique())[:25], "...")

Distinct raw descriptions: 283
['AIRTEL POSTPAID', 'AMAZON IN', 'AMAZON PRIME VIDEO', 'AMAZON SELLER SVCS', 'AMAZONIN MARKETPLACE', 'AMZN PRIME', 'AMZN-INTPYMT', 'ANI Technologies', 'ATM-WDL-HDFC-3609', 'ATM-WDL-HDFC-4942', 'ATM-WDL-HDFC-8030', 'ATM-WDL-HDFC-8253', 'ATM-WDL-HDFC-9140', 'ATM-WDL-ICICI-3918', 'ATM-WDL-ICICI-4172', 'ATM-WDL-ICICI-4739', 'ATM-WDL-ICICI-5025', 'ATM-WDL-ICICI-6478', 'ATM-WDL-ICICI-9135', 'ATM-WDL-SBI-0237', 'ATM-WDL-SBI-0279', 'ATM-WDL-SBI-0874', 'ATM-WDL-SBI-4080', 'ATM-WDL-SBI-4084', 'ATM-WDL-SBI-5715'] ...


In [8]:
VENDOR_KEYWORDS = {
    "TechCrush Labs (Salary)": ["TECHCRUSH"],
    "Landlord (Rent)":  ["RENT-LANDLORD"],
    "Instamart":        ["INSTAMART"],
    "Swiggy":           ["SWIGGY", "BUNDL"],            # BUNDL Tech = Swiggy's parent company
    "Zomato":           ["ZOMATO"],
    "Blinkit":          ["BLINKIT", "GROFERS"],         # Grofers = Blinkit's old name
    "Zepto":            ["ZEPTO"],
    "Amazon Prime":     ["PRIME"],
    "Amazon":           ["AMAZON", "AMZN"],
    "Flipkart":         ["FLIPKART", "FKART", "FSN"],
    "Myntra":           ["MYNTRA"],
    "Nykaa":            ["NYKAA"],
    "Ola":              ["OLA", "ANI TECH"],            # ANI Technologies = Ola
    "Uber":             ["UBER", "ROPPEN"],             # Roppen Transportation = Uber's India entity
    "Rapido":           ["RAPIDO"],
    "BMTC":             ["BMTC", "TUMMOC"],
    "Starbucks":        ["STARBUCKS"],
    "Cafe Coffee Day":  ["COFFEE DAY", "CCD"],
    "Third Wave Coffee":["THIRD WAVE", "THIRDWAVE", "TWC"],
    "Empire Restaurant":["EMPIRE"],
    "Meghana Foods":    ["MEGHANA"],
    "Truffles":         ["TRUFFLES"],
    "Dineout":          ["DINEOUT"],
    "Local Restaurant": ["RESTAURANT"],
    "Netflix":          ["NETFLIX"],
    "Spotify":          ["SPOTIFY"],
    "Disney+ Hotstar":  ["HOTSTAR", "STAR INDIA"],
    "BESCOM":           ["BESCOM", "ELEC SUPPLY"],
    "BWSSB":            ["BWSSB"],
    "Jio":              ["JIO"],
    "Airtel":           ["AIRTEL"],
    "Vi":               ["VI POSTPAID", "VI-RECHARGE", "VODAFONE"],
    "BigBasket":        ["BIGBASKET", "INNOVATIVE RETAIL"],
    "DMart":            ["DMART", "AVENUE SUPERMARTS"],
    "KiranaKart":       ["KIRANAKART"],
    "Zerodha":          ["ZERODHA"],
    "Groww":            ["GROWW", "NEXTBILLION"],
    "BPCL":             ["BPCL"],
    "HP Petrol":        ["HP PETROL"],
    "Indian Oil":       ["INDIAN OIL", "IOC"],
    "BookMyShow":       ["BMS MOVIE", "BIGTREE", "BOOKMYSHOW"],
}
FRIENDS = {"AMAN", "ANKIT", "KARAN", "NEHA", "PRIYA", "SNEHA", "VIKAS"}   # first names seen in UPI-<NAME>-<digits>@bank

def extract_vendor(desc):
    d = desc.upper()
    if d.startswith("ATM-WDL"):
        return "Cash Withdrawal"
    parts = d.split("-")
    if d.startswith("UPI-") and len(parts) > 1 and parts[1] in FRIENDS:
        return "P2P Transfer"
    for vendor, keywords in VENDOR_KEYWORDS.items():
        for kw in keywords:
            if kw in d:
                return vendor
    return "Uncategorised"

df["vendor_clean"] = df["Description"].apply(extract_vendor)

print("Canonical vendors found:", df["vendor_clean"].nunique())
print("Uncategorised rows     :", int((df["vendor_clean"] == "Uncategorised").sum()))
print()
print(df["vendor_clean"].value_counts().head(10))

Canonical vendors found: 43
Uncategorised rows     : 0

vendor_clean
Swiggy       176
Zomato       121
Uber          85
Amazon        76
Ola           69
Instamart     67
Zepto         58
Blinkit       55
Flipkart      50
Starbucks     42
Name: count, dtype: int64


In [9]:
VENDOR_TO_CATEGORY = {
    "TechCrush Labs (Salary)": "Income",
    "Landlord (Rent)": "Rent",
    "Swiggy": "Food Delivery", "Zomato": "Food Delivery",
    "Instamart": "Quick Commerce", "Blinkit": "Quick Commerce", "Zepto": "Quick Commerce",
    "Amazon": "Ecommerce", "Flipkart": "Ecommerce", "Myntra": "Ecommerce", "Nykaa": "Ecommerce",
    "Ola": "Transport", "Uber": "Transport", "Rapido": "Transport", "BMTC": "Transport",
    "Starbucks": "Cafe", "Cafe Coffee Day": "Cafe", "Third Wave Coffee": "Cafe",
    "Empire Restaurant": "Restaurants", "Meghana Foods": "Restaurants", "Truffles": "Restaurants",
    "Dineout": "Restaurants", "Local Restaurant": "Restaurants",
    "Netflix": "Subscriptions", "Spotify": "Subscriptions", "Disney+ Hotstar": "Subscriptions",
    "Amazon Prime": "Subscriptions",
    "BESCOM": "Utilities", "BWSSB": "Utilities", "Jio": "Utilities", "Airtel": "Utilities", "Vi": "Utilities",
    "BigBasket": "Groceries", "DMart": "Groceries", "KiranaKart": "Groceries",
    "Zerodha": "Investments", "Groww": "Investments",
    "BPCL": "Fuel", "HP Petrol": "Fuel", "Indian Oil": "Fuel",
    "BookMyShow": "Entertainment",
    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",
}
df["category"] = df["vendor_clean"].map(VENDOR_TO_CATEGORY).fillna("Uncategorised")

print(df["category"].value_counts())

category
Food Delivery        297
Transport            232
Quick Commerce       180
Ecommerce            162
Cafe                  99
Restaurants           73
Subscriptions         59
Groceries             54
Utilities             43
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64


In [10]:
credits = df[df["type"] == "credit"]
debits = df[df["type"] == "debit"]
spend = debits[debits["category"] != "Personal Transfer"]       # basis for % calculations

total_credits = credits["amount"].sum()
total_debits = debits["amount"].sum()
spend_total = spend["amount"].sum()
net_change = total_credits - total_debits
savings_rate = net_change / total_credits * 100

cat_totals = spend.groupby("category")["amount"].sum().sort_values(ascending=False)
vendor_stats = spend.groupby("vendor_clean")["amount"].agg(["sum", "count"]).sort_values("sum", ascending=False)

print(divider())
print(" EXECUTIVE SUMMARY")
print(f"  Total credits  : {inr(total_credits)}")
print(f"  Total debits   : {inr(total_debits)}")
print(f"  Net change     : {inr(net_change)} ({'overspending' if net_change < 0 else 'saving'})")
print(f"  Savings rate   : {savings_rate:.1f}%")
print(f"  Transactions   : {len(df):,}")
print(f"  Unique vendors : {df['vendor_clean'].nunique()}")
print()
print(" TOP 5 CATEGORIES (% of debits, ex-P2P)")
for cat, val in cat_totals.head(5).items():
    print(f"  {cat:<16}{bar(val, cat_totals.iloc[0], 20):<21}{val / spend_total * 100:5.1f}%  {inr(val):>14}")
print()
print(" TOP 5 VENDORS")
for v, r in vendor_stats.head(5).iterrows():
    print(f"  {v:<18}{inr(r['sum']):>14}   ({int(r['count'])} txns)")
print(divider())

 EXECUTIVE SUMMARY
  Total credits  : Rs. 5,09,774
  Total debits   : Rs. 16,78,901
  Net change     : -Rs. 11,69,127 (overspending)
  Savings rate   : -229.3%
  Transactions   : 1,310
  Unique vendors : 43

 TOP 5 CATEGORIES (% of debits, ex-P2P)
  Ecommerce       ####################  35.9%    Rs. 5,93,769
  Investments     ########              15.0%    Rs. 2,48,160
  Food Delivery   ####                   7.8%    Rs. 1,29,054
  Restaurants     ####                   7.1%    Rs. 1,17,737
  Rent            ####                   6.5%    Rs. 1,08,000

 TOP 5 VENDORS
  Amazon              Rs. 3,18,422   (76 txns)
  Zerodha             Rs. 2,10,000   (14 txns)
  Flipkart            Rs. 1,81,806   (50 txns)
  Landlord (Rent)     Rs. 1,08,000   (6 txns)
  Swiggy                Rs. 73,738   (176 txns)


In [11]:
expected = {"Total credits": 509774, "Total debits": 826679, "Savings rate %": -62.2,
            "Food Delivery share %": 23.0, "Quick Commerce share %": 19.5,
            "Ecommerce share %": 16.8, "Investments share %": 16.4}
actual = {"Total credits": total_credits, "Total debits": total_debits, "Savings rate %": savings_rate,
          "Food Delivery share %": cat_totals.get("Food Delivery", 0) / spend_total * 100,
          "Quick Commerce share %": cat_totals.get("Quick Commerce", 0) / spend_total * 100,
          "Ecommerce share %": cat_totals.get("Ecommerce", 0) / spend_total * 100,
          "Investments share %": cat_totals.get("Investments", 0) / spend_total * 100}
print(f"{'Metric':<24}{'Brief':>12}{'This file':>14}{'Match (+/-5%)':>16}")
for k in expected:
    e, a = expected[k], actual[k]
    ok = abs(a - e) <= abs(e) * 0.05
    print(f"{k:<24}{e:>12,.1f}{a:>14,.1f}{'YES' if ok else 'NO':>16}")

Metric                         Brief     This file   Match (+/-5%)
Total credits              509,774.0     509,774.0             YES
Total debits               826,679.0   1,678,901.0              NO
Savings rate %                 -62.2        -229.3              NO
Food Delivery share %           23.0           7.8              NO
Quick Commerce share %          19.5           5.5              NO
Ecommerce share %               16.8          35.9              NO
Investments share %             16.4          15.0              NO


In [12]:
MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun"}
month_pivot = spend.pivot_table(values="amount", index="category", columns="month", aggfunc="sum").fillna(0)
month_pivot.columns = [MONTH_NAMES[m] for m in month_pivot.columns]
print((month_pivot.round(0)).astype(int).to_string())

# NumPy % change first month -> last month
arr = month_pivot.to_numpy()
first, last = arr[:, 0], arr[:, -1]
growth = np.where(first > 0, (last - first) / np.where(first > 0, first, 1) * 100, np.nan)
growth_s = pd.Series(growth, index=month_pivot.index).dropna()

print()
print(f"Biggest growth  Jan->Jun: {growth_s.idxmax()} ({growth_s.max():+.0f}%)")
print(f"Biggest decline Jan->Jun: {growth_s.idxmin()} ({growth_s.min():+.0f}%)")

print()
print(" MONTHLY TREND - Food Delivery")
fd = month_pivot.loc["Food Delivery"]
for m, v in fd.items():
    print(f"  {m}  {inr(v):>12}  {bar(v, fd.max(), 30)}")

                   Jan    Feb     Mar    Apr    May     Jun
category                                                   
Cafe              3690   4273    5448   6564   5668    5802
Cash Withdrawal   2000   5000    8000   5500   8000   17000
Ecommerce        97134  92773  103772  68098  95001  136991
Entertainment     1263    474    2418   2224      0    1914
Food Delivery    20890  21452   20850  23054  22167   20641
Fuel             30322   2079   26164  18718   9138    2882
Groceries        17649   8571    6971  13061  10336    7721
Investments      38432  15000   68644  54126  48628   23330
Quick Commerce   12797  17465   17297  15259  14570   13377
Rent             18000  18000   18000  18000  18000   18000
Restaurants      16320  21772   28313   7711  22286   21335
Subscriptions     5310   7296    8683   4321   3782    5055
Transport         9951   8761    6126   8252  11978    6538
Utilities         9157   6870    5497   5831   4340   10219

Biggest growth  Jan->Jun: Cash Withdraw

In [13]:
categories = sorted(spend["category"].unique())
grid = np.zeros((len(categories), 24), dtype=int)                  # NumPy 2D: category x hour (txn counts)
for i, c in enumerate(categories):
    counts = spend[spend["category"] == c]["hour"].value_counts()
    for h, n in counts.items():
        grid[i, h] = n

shades = " .:-=+*#%@"
print("HOUR ->  " + "".join(str(h % 10) for h in range(24)) + "   (digits = hour of day, 0-23)")
for i, c in enumerate(categories):
    row = grid[i]
    mx = row.max() if row.max() > 0 else 1
    cells_ = "".join(shades[int(round((len(shades) - 1) * v / mx))] for v in row)
    print(f"{c:<17}{cells_}   peak {int(row.argmax()):02d}:00")

HOUR ->  012345678901234567890123   (digits = hour of day, 0-23)
Cafe             :: .  ..*+@=---+*#+.-  .   peak 10:00
Cash Withdrawal          *@**  * - - --*    peak 09:00
Ecommerce        #=-***+--#+@*=#=#*#-**+#   peak 11:00
Entertainment    ==     =@       @ @=@ =    peak 08:00
Food Delivery     : .::  :::=+-:=::#%@**:   peak 20:00
Fuel             *- ---  -* *@--*@ -*-* -   peak 12:00
Groceries        -:-@: * -*%=-:* * -:=%::   peak 03:00
Investments      ::  -:*  :@ :     ::: :-   peak 10:00
Quick Commerce   :..... .:#=-#=-==:*%@%-:   peak 20:00
Rent                        * @-            peak 13:00
Restaurants      .... :. :.::=+..=:#=@=:.   peak 20:00
Subscriptions    ..:#@==--:-=-..: ..: . .   peak 04:00
Transport        . ...-:+*++-=*=+++++@=-:   peak 20:00
Utilities        === =  ###::@=@==:@:= ::   peak 12:00


In [14]:
print("Raw Time  ->  hour extracted")
for t in df["Time"].sample(5, random_state=3):
    print(f"  {t}   ->   {int(t[:2])}")
print("Hour distribution is correct; all 24 hours present:", sorted(df["hour"].unique()) == list(range(24)))
print()

def share_in_window(frame, start_hour, end_hour):
    # window may wrap past midnight, e.g. 21 -> 2 means 21:00 .. 01:59
    if start_hour < end_hour:
        mask = (frame["hour"] >= start_hour) & (frame["hour"] < end_hour)
    else:
        mask = (frame["hour"] >= start_hour) | (frame["hour"] < end_hour)
    return mask.mean() * 100

fd_txn = spend[spend["category"] == "Food Delivery"]
cafe_txn = spend[spend["category"] == "Cafe"]
qc_txn = spend[spend["category"] == "Quick Commerce"]
swiggy = spend[spend["vendor_clean"] == "Swiggy"]

print(f"Food Delivery 21:00-02:00 : {share_in_window(fd_txn, 21, 2):5.1f}% of {len(fd_txn)} orders")
print(f"Food Delivery 21:00-01:00 : {share_in_window(fd_txn, 21, 1):5.1f}%")
print(f"Swiggy        21:00-01:00 : {share_in_window(swiggy, 21, 1):5.1f}% of {len(swiggy)} orders")
print(f"Cafe          08:00-11:00 : {share_in_window(cafe_txn, 8, 11):5.1f}% of {len(cafe_txn)} visits")
print(f"Quick Commerce 21:00-02:00: {share_in_window(qc_txn, 21, 2):5.1f}% of {len(qc_txn)} orders")
print()
print("Share of an even spread across 24h would be: 5h/24h = 20.8% for a 5-hour window")

# BONUS: day-of-week (spend ex rent / investments / cash, i.e. everyday spending)
everyday = spend[~spend["category"].isin(["Rent", "Investments", "Cash Withdrawal"])]
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = everyday.groupby("day_of_week")["amount"].agg(["sum", "count"]).reindex(order)
print()
print(" EVERYDAY SPEND BY DAY OF WEEK")
for d, r in dow.iterrows():
    print(f"  {d:<10}{inr(r['sum']):>14}  {bar(r['sum'], dow['sum'].max(), 28)}")
wk_day = everyday[~everyday["day_of_week"].isin(["Saturday", "Sunday"])]["amount"].sum() / 5
wk_end = everyday[everyday["day_of_week"].isin(["Saturday", "Sunday"])]["amount"].sum() / 2
print(f"\nAverage per weekday: {inr(wk_day)}  |  per weekend day: {inr(wk_end)}  ->  weekend is {(wk_end / wk_day - 1) * 100:+.0f}% vs weekday")

Raw Time  ->  hour extracted
  21:42   ->   21
  18:35   ->   18
  17:55   ->   17
  12:51   ->   12
  10:58   ->   10
Hour distribution is correct; all 24 hours present: True

Food Delivery 21:00-02:00 :  20.5% of 297 orders
Food Delivery 21:00-01:00 :  18.2%
Swiggy        21:00-01:00 :  19.3% of 176 orders
Cafe          08:00-11:00 :  29.3% of 99 visits
Quick Commerce 21:00-02:00:  20.0% of 180 orders

Share of an even spread across 24h would be: 5h/24h = 20.8% for a 5-hour window

 EVERYDAY SPEND BY DAY OF WEEK
  Monday      Rs. 1,87,733  ####################
  Tuesday     Rs. 1,92,651  #####################
  Wednesday   Rs. 2,58,606  ############################
  Thursday    Rs. 1,43,717  ################
  Friday      Rs. 1,31,969  ##############
  Saturday    Rs. 1,74,382  ###################
  Sunday      Rs. 1,63,584  ##################

Average per weekday: Rs. 1,82,935  |  per weekend day: Rs. 1,68,983  ->  weekend is -8% vs weekday


In [15]:
d = debits[debits["category"] != "Personal Transfer"].copy()
cat_mean = d.groupby("category")["amount"].transform("mean")
cat_std = d.groupby("category")["amount"].transform("std")
d["z_score"] = (d["amount"] - cat_mean) / cat_std.where(cat_std > 0)          # std == 0 -> NaN, not flagged

anomalies = d[d["z_score"] > 2].sort_values("z_score", ascending=False)
print(f"Flagged {len(anomalies)} anomalies (z > 2) out of {len(d)} debit transactions "
      f"({len(anomalies) / len(d) * 100:.1f}%)")
print()
print(f"{'Date':<8}{'Vendor':<20}{'Category':<17}{'Amount':>14}{'z':>7}")
for _, r in anomalies.head(5).iterrows():
    print(f"{r['date'].strftime('%d %b'):<8}{r['vendor_clean']:<20}{r['category']:<17}{inr(r['amount']):>14}{r['z_score']:>7.1f}")
print()
print("Anomalies by category:")
print(anomalies["category"].value_counts().to_string())

Flagged 28 anomalies (z > 2) out of 1286 debit transactions (2.2%)

Date    Vendor              Category                 Amount      z
26 Jun  Amazon              Ecommerce            Rs. 22,008    4.0
07 Feb  Amazon              Ecommerce            Rs. 21,986    4.0
26 Feb  Local Restaurant    Restaurants           Rs. 8,383    3.9
22 Jun  Dineout             Restaurants           Rs. 7,935    3.6
31 Mar  Meghana Foods       Restaurants           Rs. 7,931    3.6

Anomalies by category:
category
Ecommerce        15
Restaurants       5
Groceries         3
Subscriptions     3
Fuel              2


In [16]:
def pct_of_spend(*cats):
    return spend[spend["category"].isin(cats)]["amount"].sum() / spend_total * 100

def is_foodie():
    p = pct_of_spend("Food Delivery", "Restaurants", "Cafe");  return p > 25, f"{p:.1f}% on food"
def is_quick_commerce_junkie():
    p = pct_of_spend("Quick Commerce");                        return p > 15, f"{p:.1f}% on Q-com"
def is_shopaholic():
    p = pct_of_spend("Ecommerce");                             return p > 15, f"{p:.1f}% on e-commerce"
def is_investor():
    p = pct_of_spend("Investments");                           return p > 15, f"{p:.1f}% on investments"
def is_late_night_snacker():
    p = share_in_window(fd_txn, 21, 2);                        return p > 50, f"{p:.0f}% of food delivery 21:00-02:00"
def is_cab_commuter():
    p = pct_of_spend("Transport");                             return p > 10, f"{p:.1f}% on transport"
def is_subscription_lover():
    n = spend[spend["category"] == "Subscriptions"]["vendor_clean"].nunique()
    return n >= 5, f"{n} distinct subscription vendors"
def is_yolo_spender():
    return savings_rate < 10, f"savings rate {savings_rate:.0f}%"
def is_disciplined_saver():
    return savings_rate > 40, f"savings rate {savings_rate:.0f}%"

# BONUS invented archetype (Bengaluru): THE COFFEE-SHOP HOPPER
#   Rule: uses 3+ distinct cafe chains AND averages 2+ cafe visits per week.
#   Why: Bengaluru's cafe culture (Third Wave, Starbucks, CCD) means a hopper spreads visits across chains.
def is_coffee_shop_hopper():
    cafes = spend[spend["category"] == "Cafe"]
    weeks = (df["date"].max() - df["date"].min()).days / 7
    per_week = len(cafes) / weeks
    n = cafes["vendor_clean"].nunique()
    return (n >= 3 and per_week >= 2), f"{n} cafe chains, {per_week:.1f} visits/week"

ARCHETYPES = [
    ("THE FOODIE", is_foodie), ("THE QUICK COMMERCE JUNKIE", is_quick_commerce_junkie),
    ("THE SHOPAHOLIC", is_shopaholic), ("THE INVESTOR", is_investor),
    ("THE LATE-NIGHT SNACKER", is_late_night_snacker), ("THE CAB COMMUTER", is_cab_commuter),
    ("THE SUBSCRIPTION LOVER", is_subscription_lover), ("THE YOLO SPENDER", is_yolo_spender),
    ("THE DISCIPLINED SAVER", is_disciplined_saver),
    ("THE COFFEE-SHOP HOPPER (invented)", is_coffee_shop_hopper),
]
matched = []
print(f"{'Archetype':<38}{'Match':<8}Evidence")
for name, fn in ARCHETYPES:
    ok, evidence = fn()
    print(f"{name:<38}{'YES' if ok else 'no':<8}{evidence}")
    if ok: matched.append((name, evidence))

Archetype                             Match   Evidence
THE FOODIE                            no      16.8% on food
THE QUICK COMMERCE JUNKIE             no      5.5% on Q-com
THE SHOPAHOLIC                        YES     35.9% on e-commerce
THE INVESTOR                          YES     15.0% on investments
THE LATE-NIGHT SNACKER                no      21% of food delivery 21:00-02:00
THE CAB COMMUTER                      no      3.1% on transport
THE SUBSCRIPTION LOVER                no      4 distinct subscription vendors
THE YOLO SPENDER                      YES     savings rate -229%
THE DISCIPLINED SAVER                 no      savings rate -229%
THE COFFEE-SHOP HOPPER (invented)     YES     3 cafe chains, 3.8 visits/week


In [17]:
unmapped = df[df["vendor_clean"] == "Uncategorised"]["Description"].unique()
print(f"Descriptions the extractor could not map: {len(unmapped)}")
for u in unmapped: print("  -", u)

# --- Forecast: rolling 3-month average per category, NumPy only ---
print()
print(" NEXT-MONTH (JULY) PROJECTION = mean of last 3 months, per category")
last3 = month_pivot.to_numpy()[:, -3:]
proj = last3.mean(axis=1)
order_idx = np.argsort(-proj)
for i in order_idx[:8]:
    print(f"  {month_pivot.index[i]:<16}{inr(proj[i]):>14}")
print(f"  {'TOTAL':<16}{inr(proj.sum()):>14}   (vs. average monthly salary {inr(total_credits / 6)})")

Descriptions the extractor could not map: 0

 NEXT-MONTH (JULY) PROJECTION = mean of last 3 months, per category
  Ecommerce         Rs. 1,00,030
  Investments         Rs. 42,028
  Food Delivery       Rs. 21,954
  Rent                Rs. 18,000
  Restaurants         Rs. 17,111
  Quick Commerce      Rs. 14,402
  Groceries           Rs. 10,373
  Fuel                Rs. 10,246
  TOTAL             Rs. 2,71,806   (vs. average monthly salary Rs. 84,962)


In [18]:
def print_report():
    W = 64
    print("=" * W)
    print(" SpendDNA REPORT - RAHUL SHARMA")
    first_d, last_d = df["date"].min(), df["date"].max()
    print(f" 6 months - {len(df):,} transactions - {first_d.strftime('%b')} to {last_d.strftime('%b %Y')}")
    print("=" * W)
    print(" EXECUTIVE SUMMARY")
    print(f"  Total credits  : {inr(total_credits)}")
    print(f"  Total debits   : {inr(total_debits)}")
    print(f"  Net change     : {inr(net_change)} ({'overspending' if net_change < 0 else 'saving'})")
    flag = "BURNING SAVINGS" if savings_rate < 0 else ("TIGHT" if savings_rate < 20 else "HEALTHY")
    print(f"  Savings rate   : {savings_rate:.1f}% ({flag})")
    print(f"  Transactions   : {len(df):,}")
    print(f"  Unique vendors : {df['vendor_clean'].nunique()}")
    print()
    print(" TOP CATEGORIES (% of debit total, ex-P2P)")
    for cat, val in cat_totals.head(5).items():
        print(f"  {cat:<16}{bar(val, cat_totals.iloc[0], 18):<19}{val / spend_total * 100:5.1f}%  {inr(val):>14}")
    print()
    print(" TOP VENDORS")
    for v, r in vendor_stats.head(5).iterrows():
        print(f"  {v:<18}{inr(r['sum']):>14}  ({int(r['count']):>3} txns)")
    print()
    print(" TIME-OF-DAY PATTERNS")
    print(f"  Food Delivery 21:00-02:00 : {share_in_window(fd_txn, 21, 2):.0f}% of orders (even spread would be 21%)")
    print(f"  Cafe 08:00-11:00          : {share_in_window(cafe_txn, 8, 11):.0f}% of visits")
    print(f"  Quick Commerce 21:00-02:00: {share_in_window(qc_txn, 21, 2):.0f}% of orders")
    print()
    print(" MONTHLY TREND (Food Delivery)")
    for m, v in fd.items():
        print(f"  {m}  {inr(v):>12}  {bar(v, fd.max(), 24)}")
    print()
    print(" TOP ANOMALIES (z-score within category)")
    for _, r in anomalies.head(3).iterrows():
        print(f"  {r['date'].strftime('%d %b')} - {r['vendor_clean']:<12}{inr(r['amount']):>14}  (z={r['z_score']:.1f})")
    print(f"  ... {len(anomalies)} flagged in total")
    print()
    print(" SPENDING ARCHETYPES")
    for name, evidence in matched:
        print(f"  -> {name} ({evidence})")
    print("=" * W)

print_report()

 SpendDNA REPORT - RAHUL SHARMA
 6 months - 1,310 transactions - Jan to Jun 2024
 EXECUTIVE SUMMARY
  Total credits  : Rs. 5,09,774
  Total debits   : Rs. 16,78,901
  Net change     : -Rs. 11,69,127 (overspending)
  Savings rate   : -229.3% (BURNING SAVINGS)
  Transactions   : 1,310
  Unique vendors : 43

 TOP CATEGORIES (% of debit total, ex-P2P)
  Ecommerce       ##################  35.9%    Rs. 5,93,769
  Investments     ########            15.0%    Rs. 2,48,160
  Food Delivery   ####                 7.8%    Rs. 1,29,054
  Restaurants     ####                 7.1%    Rs. 1,17,737
  Rent            ###                  6.5%    Rs. 1,08,000

 TOP VENDORS
  Amazon              Rs. 3,18,422  ( 76 txns)
  Zerodha             Rs. 2,10,000  ( 14 txns)
  Flipkart            Rs. 1,81,806  ( 50 txns)
  Landlord (Rent)     Rs. 1,08,000  (  6 txns)
  Swiggy                Rs. 73,738  (176 txns)

 TIME-OF-DAY PATTERNS
  Food Delivery 21:00-02:00 : 21% of orders (even spread would be 21%)
  Cafe 